# Text Pre-processing Pipelines

This notebook contains two data pre-processing programs:
1. **NLTK Pre-processing Pipeline**
2. **SpaCy Pre-processing Pipeline**


## 1. NLTK Pre-processing Pipeline


In [1]:
import re
import string
import nltk
from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords, wordnet
from nltk.stem import PorterStemmer, SnowballStemmer, LancasterStemmer, WordNetLemmatizer
from nltk.tag import pos_tag

for resource in ['punkt', 'stopwords', 'wordnet', 'averaged_perceptron_tagger', 'omw-1.4']:
    nltk.download(resource, quiet=True)

text = """The 3 artificial intelligence engineers were analyzing 15 massive datasets at Google HQ! 
Dr. Smith and John Doe developed newer, faster algorithms which are performing better than older methods. 
Visit https://deepmind.google/ or contact research-team@deepmind.com for $500 travel grants. 
He was studying, running, and writing multiple complex queries effectively yesterday."""

print("Original Text:")
print(text)
print("-" * 50)

lower_text = text.lower()

cleaned_text = re.sub(r'https?://\S+|www\.\S+', '', text)
cleaned_text = re.sub(r'\S+@\S+', '', cleaned_text)
cleaned_text = re.sub(r'[^a-zA-Z\s]', ' ', cleaned_text)
cleaned_text = re.sub(r'\s+', ' ', cleaned_text).strip()

sentences = sent_tokenize(text)
print("Sentences:")
for s in sentences:
    print("-", s)
print("-" * 50)

tokens = word_tokenize(cleaned_text.lower())
print("Word Tokens:")
print(tokens)
print("-" * 50)

stop_words = set(stopwords.words('english'))
filtered_tokens = [w for w in tokens if w not in stop_words and w not in string.punctuation and len(w) > 1]
print("Tokens after removing punctuation and stopwords:")
print(filtered_tokens)
print("-" * 50)

porter = PorterStemmer()
snowball = SnowballStemmer('english')
lancaster = LancasterStemmer()

sample_words = ["analyzing", "engineers", "developed", "faster", "algorithms", "performing", "better", "studying", "running", "writing", "queries"]
print(f"{'Word':<15} {'Porter':<15} {'Snowball':<15} {'Lancaster':<15}")
for w in sample_words:
    print(f"{w:<15} {porter.stem(w):<15} {snowball.stem(w):<15} {lancaster.stem(w):<15}")
print("-" * 50)

lemmatizer = WordNetLemmatizer()
simple_lemmas = [lemmatizer.lemmatize(w) for w in filtered_tokens]

pos_tags = pos_tag(filtered_tokens)
print("POS Tags:")
print(pos_tags)
print("-" * 50)

def get_wordnet_pos(tag):
    if tag.startswith('J'):
        return wordnet.ADJ
    elif tag.startswith('V'):
        return wordnet.VERB
    elif tag.startswith('N'):
        return wordnet.NOUN
    elif tag.startswith('R'):
        return wordnet.ADV
    return wordnet.NOUN

pos_lemmas = [lemmatizer.lemmatize(w, get_wordnet_pos(t)) for w, t in pos_tags]

print(f"{'Token':<15} {'POS Tag':<10} {'Simple Lemma':<15} {'POS-Aware Lemma':<15}")
for (w, t), s_l, p_l in zip(pos_tags[:8], simple_lemmas[:8], pos_lemmas[:8]):
    print(f"{w:<15} {t:<10} {s_l:<15} {p_l:<15}")
print("-" * 50)

def nltk_pipeline(input_text):
    text_clean = re.sub(r'https?://\S+|www\.\S+', '', input_text.lower())
    text_clean = re.sub(r'\S+@\S+', '', text_clean)
    text_clean = re.sub(r'[^a-z\s]', ' ', text_clean)
    text_clean = re.sub(r'\s+', ' ', text_clean).strip()
    
    words = word_tokenize(text_clean)
    stops = set(stopwords.words('english'))
    words = [w for w in words if w not in stops and w not in string.punctuation and len(w) > 1]
    
    tags = pos_tag(words)
    wnl = WordNetLemmatizer()
    lemmas = [wnl.lemmatize(w, get_wordnet_pos(t)) for w, t in tags]
    
    return lemmas

result = nltk_pipeline(text)
print("Before vs After Pre-processing:")
print("Original Text Length:", len(text))
print("Processed Tokens Count:", len(result))
print("Final Processed Output:")
print(result)


Original Text:
The 3 artificial intelligence engineers were analyzing 15 massive datasets at Google HQ! 
Dr. Smith and John Doe developed newer, faster algorithms which are performing better than older methods. 
Visit https://deepmind.google/ or contact research-team@deepmind.com for $500 travel grants. 
He was studying, running, and writing multiple complex queries effectively yesterday.
--------------------------------------------------
Sentences:
- The 3 artificial intelligence engineers were analyzing 15 massive datasets at Google HQ!
- Dr. Smith and John Doe developed newer, faster algorithms which are performing better than older methods.
- Visit https://deepmind.google/ or contact research-team@deepmind.com for $500 travel grants.
- He was studying, running, and writing multiple complex queries effectively yesterday.
--------------------------------------------------
Word Tokens:
['the', 'artificial', 'intelligence', 'engineers', 'were', 'analyzing', 'massive', 'datasets', 'at',

POS Tags:
[('artificial', 'JJ'), ('intelligence', 'NN'), ('engineers', 'NNS'), ('analyzing', 'VBG'), ('massive', 'JJ'), ('datasets', 'NNS'), ('google', 'VBP'), ('hq', 'JJ'), ('dr', 'NN'), ('smith', 'NN'), ('john', 'NN'), ('doe', 'NN'), ('developed', 'VBD'), ('newer', 'JJR'), ('faster', 'RBR'), ('algorithms', 'IN'), ('performing', 'VBG'), ('better', 'JJR'), ('older', 'JJR'), ('methods', 'NNS'), ('visit', 'VBP'), ('contact', 'JJ'), ('travel', 'NN'), ('grants', 'NNS'), ('studying', 'VBG'), ('running', 'VBG'), ('writing', 'VBG'), ('multiple', 'JJ'), ('complex', 'JJ'), ('queries', 'NNS'), ('effectively', 'RB'), ('yesterday', 'NN')]
--------------------------------------------------
Token           POS Tag    Simple Lemma    POS-Aware Lemma
artificial      JJ         artificial      artificial     
intelligence    NN         intelligence    intelligence   
engineers       NNS        engineer        engineer       
analyzing       VBG        analyzing       analyze        
massive         JJ 

## 2. SpaCy Pre-processing Pipeline


In [2]:
import spacy

try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    spacy.cli.download("en_core_web_sm")
    nlp = spacy.load("en_core_web_sm")

text = """The 3 artificial intelligence engineers were analyzing 15 massive datasets at Google HQ! 
Dr. Smith and John Doe developed newer, faster algorithms which are performing better than older methods. 
Visit https://deepmind.google/ or contact research-team@deepmind.com for $500 travel grants. 
He was studying, running, and writing multiple complex queries effectively yesterday."""

doc = nlp(text)

print("Original Text:")
print(text)
print("-" * 50)

print("Sentence Segmentation:")
for s in doc.sents:
    print("-", s.text.strip())
print("-" * 50)

print("Token Attributes Sample:")
print(f"{'Text':<15} {'Lemma':<12} {'POS':<8} {'Is Stop':<10} {'Is Punct':<10} {'Is Alpha':<10}")
for token in list(doc)[:10]:
    print(f"{token.text:<15} {token.lemma_:<12} {token.pos_:<8} {str(token.is_stop):<10} {str(token.is_punct):<10} {str(token.is_alpha):<10}")
print("-" * 50)

filtered_tokens = [t.lower_ for t in doc if not t.is_stop and not t.is_punct and not t.is_space and t.is_alpha]
filtered_lemmas = [t.lemma_.lower() for t in doc if not t.is_stop and not t.is_punct and not t.is_space and t.is_alpha]

print("Filtered Tokens (lowercase, no stopwords/punct):")
print(filtered_tokens)
print("Filtered Lemmas:")
print(filtered_lemmas)
print("-" * 50)

print("Named Entities (NER):")
for ent in doc.ents:
    print(f"{ent.text:<25} {ent.label_:<12} ({spacy.explain(ent.label_)})")
print("-" * 50)

urls = [token.text for token in doc if token.like_url]
emails = [token.text for token in doc if token.like_email]
numbers = [token.text for token in doc if token.like_num or token.is_digit]

print("Pattern Detection:")
print("URLs:", urls)
print("Emails:", emails)
print("Numbers:", numbers)
print("-" * 50)

content_pos = {'NOUN', 'PROPN', 'VERB', 'ADJ'}
pos_filtered = [(t.text, t.lemma_.lower(), t.pos_) for t in doc if t.pos_ in content_pos and not t.is_stop and t.is_alpha]

print("Tokens Filtered by POS (NOUN, PROPN, VERB, ADJ):")
for word, lemma, pos in pos_filtered:
    print(f"{word:<15} -> {lemma:<15} [{pos}]")
print("-" * 50)

def spacy_pipeline(input_text, allowed_pos={'NOUN', 'PROPN', 'VERB', 'ADJ'}):
    parsed = nlp(input_text)
    return [
        t.lemma_.lower() 
        for t in parsed 
        if not t.is_stop and not t.is_punct and not t.is_space and t.is_alpha and t.pos_ in allowed_pos
    ]

result = spacy_pipeline(text)
print("Before vs After Pre-processing:")
print("Original Document Tokens:", len(doc))
print("Processed Tokens Count:", len(result))
print("Final Processed Output:")
print(result)


/Users/anshumaansoni/PycharmProjects/Natural Language Processing/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Original Text:
The 3 artificial intelligence engineers were analyzing 15 massive datasets at Google HQ! 
Dr. Smith and John Doe developed newer, faster algorithms which are performing better than older methods. 
Visit https://deepmind.google/ or contact research-team@deepmind.com for $500 travel grants. 
He was studying, running, and writing multiple complex queries effectively yesterday.
--------------------------------------------------
Sentence Segmentation:
- The 3 artificial intelligence engineers were analyzing 15 massive datasets at Google HQ!
- Dr. Smith and John Doe developed newer, faster algorithms which are performing better than older methods.
- Visit https://deepmind.google/ or contact research-team@deepmind.com for $500 travel grants.
- He was studying, running, and writing multiple complex queries effectively yesterday.
--------------------------------------------------
Token Attributes Sample:
Text            Lemma        POS      Is Stop    Is Punct   Is Alpha  
The  